# MoveBench: Probability Surfaces (Single Step)

Visualize **log movement probability** surfaces for a single step from a chosen study:

- Panel 1: "Don't move" (NDVI only)
- Panel 2: iSSF (linear)
- Panel 3: iSSF-MLP
- Panel 4: deepSSF

All panels use the **NDVI layer** as the background, and overlay the model's **log probability** over the H×W grid.

Assumptions:
- You have rasters downloaded for the study and `LAYERS_CONFIG`.
- You have extracted point feature stores + normalization stats (for iSSF feature normalization).
- You have trained checkpoints saved by `benchmark.py` (or you set `*_CKPT_PATH` manually).

Outputs:
- Saves a PNG under `pngs/` at the repo root.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# --- Repo + data root wiring ---
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "movebench").exists():
    repo_root = repo_root.parent
if not (repo_root / "movebench").exists():
    raise RuntimeError("Could not find repo root (expected a movebench/ directory). Set os.chdir(...) first.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

_default_data_root = (repo_root / "data" / "movebench").resolve()
if "MOVEBENCH_DATA_ROOT" not in os.environ and _default_data_root.exists():
    os.environ["MOVEBENCH_DATA_ROOT"] = str(_default_data_root)

print("repo_root:", repo_root)
print("MOVEBENCH_DATA_ROOT:", os.environ.get("MOVEBENCH_DATA_ROOT"))


In [ ]:
import math
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from movebench import Study
from movebench.datasets.chips import ChipDataset
from movebench.datasets.deepssf import DeepSSFDataset
from movebench.features.normalization import load_points_normalization_mean_std
from movebench.features.points import select_point_store_dir
from movebench.models.deepssf import DeepSSF
from movebench.models.ssf import LinearStepSelect, MLPStepSelect


In [ ]:
# =====================
# Parameters
# =====================
STUDY_ID = "tsavo_lions"
SPLIT = "test"  # "train" or "test"
STEP_IDX = 0    # index into ChipDataset(mode='steps')

# Layers config used for chips + (typically) for the benchmark
LAYERS_CONFIG = "configs/layers_all.yaml"

# Chip/grid size (deepSSF uses square grids)
OUTPUT_HW = (101, 101)

# Benchmark-ish movement sampler params (used to locate point feature stores)
NUM_CANDIDATES = 100
RANDOM_SEED = 0
STEP_MULTIPLIER = 2.0
STEP_Q = 0.95
GAP_Q = 0.999
BOUNDS_BUFFER_DEG = 0.01

# Checkpoints (default: where benchmark.py writes)
CKPT_DIR = repo_root / "results" / "benchmarks" / "checkpoints" / STUDY_ID
ISSF_CKPT_PATH = CKPT_DIR / "issf_linear.pt"
ISSF_MLP_CKPT_PATH = CKPT_DIR / "issf_mlp.pt"
DEEPSSF_CKPT_PATH = CKPT_DIR / "deepssf.pt"

# Visualization
OVERLAY_ALPHA = 0.55
NDVI_CMAP = "gray"
LOGP_CMAP = "magma"
OUT_DIR = (repo_root / "pngs").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("CKPT_DIR:", CKPT_DIR)


In [ ]:
# --- Load study + plan ---
study = Study(STUDY_ID)
plan = study.layer_plan(LAYERS_CONFIG)

band_ids = plan.band_ids_in_order()
NDVI_BAND_ID = "modis_vi.NDVI"
if NDVI_BAND_ID not in band_ids:
    raise KeyError(
        f"NDVI band_id {NDVI_BAND_ID!r} not found in this LayerPlan. "
        f"Available modis_vi bands: {[b for b in band_ids if b.startswith('modis_vi.')]}."
    )
ndvi_idx = band_ids.index(NDVI_BAND_ID)
print("config_hash:", plan.config_hash)
print("rasters_hash:", plan.rasters_hash)
print("num_env_channels:", plan.num_channels())
print("ndvi_idx:", ndvi_idx)


In [ ]:
# --- Select point-store sampler_hash (for normalization + movement stats) ---
# This expects you have extracted parametric point stores.
store_dir, manifest = select_point_store_dir(
    study_dir=study.study_dir,
    split=SPLIT,
    config_hash=str(plan.config_hash),
    expected_type="parametric",
    num_candidates=int(NUM_CANDIDATES),
    random_seed=int(RANDOM_SEED),
    step_multiplier=float(STEP_MULTIPLIER),
    step_q=float(STEP_Q),
    gap_q=float(GAP_Q),
    bounds_buffer_deg=float(BOUNDS_BUFFER_DEG),
)
sampler_hash = str(manifest.sampler_hash)
print("point_store_dir:", store_dir)
print("sampler_hash:", sampler_hash)

# Env normalization vectors (used by iSSF + deepSSF training)
mean_env, std_env = load_points_normalization_mean_std(
    study_dir=study.study_dir,
    plan=plan,
    sampler_hash=sampler_hash,
)

# Movement normalization stats (match iSSFDataset default)
train_store_dir = (
    study.study_dir
    / "features"
    / "points"
    / "train"
    / str(plan.config_hash)
    / str(sampler_hash)
)
train_pt = train_store_dir / "features.pt"
if not train_pt.exists():
    raise FileNotFoundError(f"Train point store not found (needed for movement normalization stats): {train_pt}")

td = torch.load(train_pt, map_location="cpu")
sl_true = td["step_length_m"][:, -1].to(torch.float64)
ta_true = td["turn_angle_rad"][:, -1].to(torch.float64)
log_sl_true = torch.log(sl_true + 1e-8)
cos_ta_true = torch.cos(ta_true)

def _mean_std(x: torch.Tensor) -> tuple[float, float]:
    m = float(x.mean().item())
    s = float(x.std(unbiased=False).item())
    if not math.isfinite(s) or s < 1e-8:
        s = 1.0
    return m, s

sl_mean, sl_std = _mean_std(sl_true)
log_sl_mean, log_sl_std = _mean_std(log_sl_true)
cos_ta_mean, cos_ta_std = _mean_std(cos_ta_true)

movement_stats = {
    "sl_mean": sl_mean,
    "sl_std": sl_std,
    "log_sl_mean": log_sl_mean,
    "log_sl_std": log_sl_std,
    "cos_ta_mean": cos_ta_mean,
    "cos_ta_std": cos_ta_std,
}
movement_stats


In [ ]:
# --- Load chip(s) for this step ---
# Raw chips: NDVI background
chips_raw = ChipDataset(
    study_dir=study.study_dir,
    trajectories_path=study.trajectories_path,
    split=SPLIT,
    plan=plan,
    mode="steps",
    output_hw=OUTPUT_HW,
    normalize=False,
    step_multiplier=float(STEP_MULTIPLIER),
    step_q=float(STEP_Q),
    gap_q=float(GAP_Q),
)

# Normalized chips: deepSSF model input
chips_norm = ChipDataset(
    study_dir=study.study_dir,
    trajectories_path=study.trajectories_path,
    split=SPLIT,
    plan=plan,
    mode="steps",
    output_hw=OUTPUT_HW,
    normalize=True,
    normalization_sampler_hash=sampler_hash,
    step_multiplier=float(STEP_MULTIPLIER),
    step_q=float(STEP_Q),
    gap_q=float(GAP_Q),
)

steps_df = chips_raw.step_starts_df
if not (0 <= int(STEP_IDX) < len(steps_df)):
    raise IndexError(f"STEP_IDX out of range: {STEP_IDX} (len={len(steps_df)})")

row = steps_df.iloc[int(STEP_IDX)]
start_bearing = float(row["start_bearing"])
timestamp = pd.Timestamp(row["timestamp"]).to_datetime64()
print("timestamp:", row["timestamp"], "start_bearing(rad):", start_bearing)

env_raw, valid_raw, y, xfm, crs = chips_raw[int(STEP_IDX)]
H, W = int(env_raw.shape[1]), int(env_raw.shape[2])
pixel_size_m = float(chips_raw.pixel_size_m)
print("env_raw:", tuple(env_raw.shape), "valid_raw:", tuple(valid_raw.shape))
print("grid:", (H, W), "pixel_size_m:", pixel_size_m, "crs:", crs)


In [ ]:
# --- Build per-pixel iSSF feature tensor ---
def _wrap_pi(x: torch.Tensor) -> torch.Tensor:
    return (x + math.pi) % (2.0 * math.pi) - math.pi


def _time_features_issf(ts: pd.Timestamp) -> torch.Tensor:
    # iSSFDataset ordering: [sin_doy, cos_doy, sin_tod, cos_tod]
    ts = pd.to_datetime(ts, utc=True)
    doy = int(ts.dayofyear)
    hour_frac = float(ts.hour) + float(ts.minute) / 60.0 + float(ts.second) / 3600.0
    sin_doy = math.sin(2.0 * math.pi * float(doy) / 365.25)
    cos_doy = math.cos(2.0 * math.pi * float(doy) / 365.25)
    sin_tod = math.sin(2.0 * math.pi * hour_frac / 24.0)
    cos_tod = math.cos(2.0 * math.pi * hour_frac / 24.0)
    return torch.tensor([sin_doy, cos_doy, sin_tod, cos_tod], dtype=torch.float32)


def build_issf_pixel_features(
    *,
    env_raw: torch.Tensor,      # (C,H,W)
    valid_raw: torch.Tensor,    # (C,H,W)
    start_bearing: float,
    timestamp: pd.Timestamp,
    mean_env: torch.Tensor,     # (C,)
    std_env: torch.Tensor,      # (C,)
    movement_stats: dict,
) -> torch.Tensor:
    C, H, W = env_raw.shape
    center_r = H // 2
    center_c = W // 2
    px = pixel_size_m

    # Flatten env
    env = env_raw.view(int(C), int(H) * int(W)).to(torch.float32)
    env_valid = valid_raw.view(int(C), int(H) * int(W)).to(torch.bool)

    # Normalize env (invalid -> 0)
    mean = mean_env.view(int(C), 1)
    std = std_env.view(int(C), 1)
    envn = torch.where(env_valid, (env - mean) / std, torch.zeros_like(env))

    # Per-pixel movement features relative to chip center (start location)
    rr = torch.arange(int(H), dtype=torch.float32).view(int(H), 1).expand(int(H), int(W))
    cc = torch.arange(int(W), dtype=torch.float32).view(1, int(W)).expand(int(H), int(W))
    dx = (cc - float(center_c)) * float(px)                # +east
    dy = (float(center_r) - rr) * float(px)                # +north
    sl = torch.sqrt(dx * dx + dy * dy).view(-1)            # (P,)
    heading = torch.atan2(dx, dy).view(-1)                 # north0
    ta = _wrap_pi(heading - float(start_bearing)).view(-1)

    log_sl = torch.log(sl + 1e-8)
    cos_ta = torch.cos(ta)

    # Normalize movement (match iSSFDataset)
    sl = (sl - float(movement_stats["sl_mean"])) / float(movement_stats["sl_std"])
    log_sl = (log_sl - float(movement_stats["log_sl_mean"])) / float(movement_stats["log_sl_std"])
    cos_ta = (cos_ta - float(movement_stats["cos_ta_mean"])) / float(movement_stats["cos_ta_std"])
    move = torch.stack([sl, log_sl, cos_ta], dim=0)        # (3,P)

    # Time features
    t = _time_features_issf(pd.Timestamp(timestamp)).view(4, 1).expand(4, int(H) * int(W))

    # Interactions (invalid env -> 0)
    def interactions(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        Ka, P = a.shape
        Kb = b.shape[0]
        return (a.unsqueeze(1) * b.unsqueeze(0)).reshape(Ka * Kb, P)

    move_env = interactions(move, envn)
    move_env_valid = env_valid.unsqueeze(0).expand(3, -1, -1).reshape(3 * int(C), int(H) * int(W))
    move_env = torch.where(move_env_valid, move_env, torch.zeros_like(move_env))

    time_env = interactions(t, envn)
    time_env_valid = env_valid.unsqueeze(0).expand(4, -1, -1).reshape(4 * int(C), int(H) * int(W))
    time_env = torch.where(time_env_valid, time_env, torch.zeros_like(time_env))

    time_move = interactions(t, move)  # always valid

    feats = torch.cat([envn, move, t, move_env, time_env, time_move], dim=0)  # (F,P)
    return feats


x_issf = build_issf_pixel_features(
    env_raw=env_raw,
    valid_raw=valid_raw,
    start_bearing=start_bearing,
    timestamp=pd.Timestamp(row["timestamp"]),
    mean_env=mean_env,
    std_env=std_env,
    movement_stats=movement_stats,
)
print("iSSF feature tensor:", tuple(x_issf.shape))


In [ ]:
# --- Load models ---
def load_ckpt(path: Path) -> dict:
    if not Path(path).exists():
        raise FileNotFoundError(f"Checkpoint not found: {path}")
    d = torch.load(str(path), map_location="cpu")
    if not isinstance(d, dict) or "state_dict" not in d:
        raise ValueError(f"Unexpected checkpoint format: {path}")
    return d


ck_issf = load_ckpt(ISSF_CKPT_PATH)
ck_mlp = load_ckpt(ISSF_MLP_CKPT_PATH)
ck_deep = load_ckpt(DEEPSSF_CKPT_PATH)

F = int(x_issf.shape[0])
issf_model = LinearStepSelect(F)
issf_model.load_state_dict(ck_issf["state_dict"], strict=True)

# Benchmark default: hidden_dims=[128], dropout=0.6
issf_mlp_model = MLPStepSelect(F, hidden_dims=(128,), dropout=0.6)
issf_mlp_model.load_state_dict(ck_mlp["state_dict"], strict=True)

# deepSSF
deep_meta = dict(ck_deep.get("meta", {}))
deep_model = DeepSSF(
    env_channels=int(plan.num_channels()),
    image_dim=int(OUTPUT_HW[0]),
    pixel_size_m=float(deep_meta.get("pixel_size_m", chips_norm.pixel_size_m)),
    device=device,
)
deep_model.load_state_dict(ck_deep["state_dict"], strict=True)

issf_model = issf_model.to(device).eval()
issf_mlp_model = issf_mlp_model.to(device).eval()
deep_model = deep_model.to(device).eval()

print("loaded:")
print(" -", ISSF_CKPT_PATH)
print(" -", ISSF_MLP_CKPT_PATH)
print(" -", DEEPSSF_CKPT_PATH)


In [ ]:
# --- Compute per-pixel log-probability surfaces ---
P = int(H) * int(W)
x = x_issf.to(device)
x_b = x.unsqueeze(0)  # (1,F,P)

with torch.no_grad():
    logits_issf = issf_model(x_b)              # (1,P)
    logits_mlp = issf_mlp_model(x_b)          # (1,P)
    logp_issf = torch.log_softmax(logits_issf, dim=1).view(int(H), int(W)).detach().cpu().numpy()
    logp_mlp = torch.log_softmax(logits_mlp, dim=1).view(int(H), int(W)).detach().cpu().numpy()

    deep_ds = DeepSSFDataset(chips_norm)
    spatial, scalars_to_grid, bearing, _target = deep_ds[int(STEP_IDX)]
    spatial = spatial.unsqueeze(0).to(device)                 # (1,C,H,W)
    scalars_to_grid = scalars_to_grid.view(1, 4).to(device)   # (1,4)
    bearing = bearing.view(1, 1).to(device)                   # (1,1)
    out = deep_model((spatial, scalars_to_grid, bearing))      # (1,H,W,2)
    scores = out[0, ..., 0] + out[0, ..., 1]                   # (H,W)
    logp_deep = torch.log_softmax(scores.view(1, -1), dim=1).view(int(H), int(W)).detach().cpu().numpy()

print("logp ranges:")
print(" issf:", float(np.nanmin(logp_issf)), float(np.nanmax(logp_issf)))
print(" mlp :", float(np.nanmin(logp_mlp)), float(np.nanmax(logp_mlp)))
print(" deep:", float(np.nanmin(logp_deep)), float(np.nanmax(logp_deep)))


In [ ]:
# --- Plot (4 panels) and save ---
ndvi = env_raw[ndvi_idx].detach().cpu().numpy().astype(np.float32, copy=False)
ndvi_valid = valid_raw[ndvi_idx].detach().cpu().numpy().astype(bool, copy=False)
ndvi_show = np.where(ndvi_valid, ndvi, np.nan)

vmin_ndvi, vmax_ndvi = np.nanpercentile(ndvi_show, [2, 98]).tolist()
stack = np.stack([logp_issf, logp_mlp, logp_deep], axis=0)
vmin_logp, vmax_logp = np.nanpercentile(stack, [1, 99]).tolist()

fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
axes = axes.reshape(2, 2)

def draw(ax, title: str, logp: np.ndarray | None):
    ax.imshow(ndvi_show, cmap=NDVI_CMAP, vmin=vmin_ndvi, vmax=vmax_ndvi)
    if logp is not None:
        im = ax.imshow(logp, cmap=LOGP_CMAP, vmin=vmin_logp, vmax=vmax_logp, alpha=float(OVERLAY_ALPHA))
    else:
        im = None
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    return im

im0 = draw(axes[0, 0], "Don't move", None)
im1 = draw(axes[0, 1], "iSSF", logp_issf)
im2 = draw(axes[1, 0], "iSSF-MLP", logp_mlp)
im3 = draw(axes[1, 1], "deepSSF", logp_deep)

# Shared colorbar for log-prob overlay
cbar = fig.colorbar(im3, ax=axes.ravel().tolist(), shrink=0.75, pad=0.02)
cbar.set_label("log movement probability (normalized over grid)")

fig.suptitle(f"{STUDY_ID} | {SPLIT} | step_idx={STEP_IDX}", y=1.02)

out_path = OUT_DIR / f"prob_surfaces_{STUDY_ID}_{SPLIT}_{int(STEP_IDX)}.png"
fig.savefig(out_path, dpi=220, bbox_inches="tight")
print("wrote:", out_path)
plt.show()
